# 02 — Data Cleaning (Phase 2)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 2: import และเรียกใช้ฟังก์ชันจาก `src/rri/cleaning/`
เท่านั้น — ไม่มี business logic ในเซลล์ตาม CLAUDE.md N2 ลำดับขั้นตอนคงที่ตามที่ตกลงไว้:

1. Extract + structural fixes (encoding, BOM, header typo, twin-column coalesce, BE→CE)
2. Harmonize category taxonomy (**ก่อน** dedup)
3. Resolve dates ← **CHECKPOINT**
4. Deduplicate
5. Remaining field-level fixes (coordinates, times, injury policy, nulls)
6. Validate + write staging

อ้างอิงข้อเท็จจริงและเหตุผลทั้งหมดจาก Phase 1: [`CLAUDE.md`](../CLAUDE.md) §6 และ
[`docs/data_quality_findings.md`](../docs/data_quality_findings.md)

## 0. Setup

In [1]:
import sys
from datetime import UTC, datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import polars as pl

from config.settings import settings
from rri.cleaning.extract import (
    add_extract_metadata,
    coalesce_twin_columns,
    convert_buddhist_year,
    pivot_rainfall_leads,
    read_raw_csv,
    rename_headers,
)
from rri.cleaning.log import CleaningLogEntry, cleaning_log_to_frame
from rri.utils.logging import configure_logging

configure_logging()
settings.ensure_writable_dirs()
RUN_TIMESTAMP = datetime.now(UTC)
cleaning_log: list[CleaningLogEntry] = []
settings.raw_dir

PosixPath('/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/00_raw')

## 1. Extract + structural fixes

เริ่มจาก extract ไฟล์ CSV ทั้ง 6 ไฟล์ (ไฟล์ accidents หลักจะผ่านอีก 5 ขั้นตอนถัดไป
ส่วนอีก 5 ไฟล์จบที่ขั้นตอนนี้) ทุกไฟล์อ่านด้วย `read_raw_csv` (encoding ตาม P7.2 —
BOM 5 ใน 6 ไฟล์, `stt_car_fuel` ไม่มี) แล้วเติม `_source_file`/`_ingested_at`/`_row_hash`

In [2]:
accidents_raw = read_raw_csv(settings.raw_dir / "thailand_accidents_2019_2026_combined.csv")
accidents_raw = add_extract_metadata(
    accidents_raw, "thailand_accidents_2019_2026_combined.csv", RUN_TIMESTAMP
)

accidents_step1, twin_logs = coalesce_twin_columns(accidents_raw)

for log_entry in twin_logs:
    cleaning_log.append(
        CleaningLogEntry(
            step="1_extract",
            rule=f"coalesce_twin_columns:{log_entry.old_column}",
            rows_before=log_entry.total_rows,
            rows_affected=log_entry.rows_from_new,
            description=(
                f"{log_entry.old_column} <- {log_entry.new_column}: "
                f"{log_entry.rows_from_old} from old, {log_entry.rows_from_new} from new, "
                f"{log_entry.rows_still_null} still null"
            ),
        )
    )

cleaning_log_to_frame(cleaning_log)

step,rule,rows_before,rows_affected,description
str,str,i64,i64,str
"""1_extract""","""coalesce_twin_columns:รถยนต์นั…",161307,8738,"""รถยนต์นั่งส่วนบุคคล <- รถยนต์น…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกไม่เกิน10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกมากกว่า10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถอื่นๆ""",161307,8738,"""รถอื่นๆ <- อื่นๆ: 133375 from …"


**ยืนยันตรงกับ Phase 1**: overlap ระหว่างคอลัมน์เก่า/ใหม่ทั้ง 4 คู่ = 0 แถว (ถ้าไม่ตรง
`coalesce_twin_columns` จะ raise ทันที ไม่ใช่ coalesce มั่ว ๆ) จำนวนแถวที่มาจากคอลัมน์ใหม่ต่อคู่
ตรงกับสัดส่วน 5.4% ที่ Phase 1 พบ

In [3]:
rainfall_raw = read_raw_csv(settings.raw_dir / "thailand_rainfall_2021_2026_combined.csv")
rainfall_raw = add_extract_metadata(
    rainfall_raw, "thailand_rainfall_2021_2026_combined.csv", RUN_TIMESTAMP
)
rainfall_renamed = rename_headers(rainfall_raw, {"Foreast_Month": "Forecast_Month"})

rainfall_step1 = pivot_rainfall_leads(rainfall_renamed)
cleaning_log.append(
    CleaningLogEntry(
        "1_extract",
        "pivot_rainfall_leads",
        rainfall_renamed.height,
        rainfall_step1.height,
        f"{rainfall_renamed.height} rows (6 leads x group) -> "
        f"{rainfall_step1.height} groups x rain_lead0..5",
    )
)
pl.DataFrame(
    [
        {
            "rows_before_pivot": rainfall_renamed.height,
            "rows_after_pivot": rainfall_step1.height,
            "columns": rainfall_step1.columns,
        }
    ]
)

rows_before_pivot,rows_after_pivot,columns
i64,i64,list[str]
30954,5159,"[""Initial_Year"", ""Initial_Month"", … ""rain_lead5""]"


**ไม่มีแถวหายไป**: `rows_before_pivot / 6 == rows_after_pivot` (6 lead-time ต่อกลุ่มกลายเป็น
1 แถว x 6 คอลัมน์ ไม่ใช่การ dedup) ยืนยันตาม P7.5

In [4]:
registration_raw = read_raw_csv(settings.raw_dir / "car_registered_new_2565_2569_combined.csv")
registration_raw = add_extract_metadata(
    registration_raw, "car_registered_new_2565_2569_combined.csv", RUN_TIMESTAMP
)
registration_step1 = convert_buddhist_year(registration_raw, "ปี พ.ศ.")
cleaning_log.append(
    CleaningLogEntry(
        "1_extract",
        "convert_buddhist_year",
        registration_step1.height,
        registration_step1.height,
        "added year_ce = ปี พ.ศ. - 543 for every row (original column kept)",
    )
)
registration_step1.select("ปี พ.ศ.", "year_ce").unique().sort("ปี พ.ศ.")

ปี พ.ศ.,year_ce
str,i64
"""2565""",2022
"""2566""",2023
"""2567""",2024
"""2568""",2025
"""2569""",2026


**ตรวจแล้ว**: พ.ศ. 2565–2569 กลายเป็น ค.ศ. 2022–2026 ตรงตามสูตร `CE = BE - 543`
คอลัมน์เดิมยังอยู่ครบ (ไม่ทำลายข้อมูลดิบ)

In [5]:
flood_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "thailand_flood_risk_areas.csv"),
    "thailand_flood_risk_areas.csv",
    RUN_TIMESTAMP,
)
heavy_rain_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "thailand_heavy_rain_R35mm.csv"),
    "thailand_heavy_rain_R35mm.csv",
    RUN_TIMESTAMP,
)
stt_car_fuel_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "stt_car_fuel_at_25690228.csv"),
    "stt_car_fuel_at_25690228.csv",
    RUN_TIMESTAMP,
)

pl.DataFrame(
    [
        {"file": "flood_risk", "rows": flood_raw.height, "cols": flood_raw.width},
        {"file": "heavy_rain_r35mm", "rows": heavy_rain_raw.height, "cols": heavy_rain_raw.width},
        {"file": "stt_car_fuel", "rows": stt_car_fuel_raw.height, "cols": stt_car_fuel_raw.width},
    ]
)

file,rows,cols
str,i64,i64
"""flood_risk""",27023,16
"""heavy_rain_r35mm""",3465,7
"""stt_car_fuel""",4643,8


ไฟล์อีก 3 ไฟล์นี้ไม่มี structural fix เฉพาะใน Step 1 (นอกจาก encoding/metadata) —
เขียนลง staging ตรง ๆ

In [6]:
accidents_step1.write_parquet(settings.staging_dir / "accidents.parquet")
rainfall_step1.write_parquet(settings.staging_dir / "rainfall.parquet")
registration_step1.write_parquet(settings.staging_dir / "car_registered_new.parquet")
flood_raw.write_parquet(settings.staging_dir / "flood_risk.parquet")
heavy_rain_raw.write_parquet(settings.staging_dir / "heavy_rain_r35mm.parquet")
stt_car_fuel_raw.write_parquet(settings.staging_dir / "stt_car_fuel.parquet")

sorted(p.name for p in settings.staging_dir.glob("*.parquet"))

['accidents.parquet',
 'car_registered_new.parquet',
 'flood_risk.parquet',
 'heavy_rain_r35mm.parquet',
 'rainfall.parquet',
 'stt_car_fuel.parquet']

**Step 1 เสร็จสมบูรณ์** — เขียนไฟล์ทั้ง 6 ลง `data/10_staging/` แล้ว `accidents.parquet`
จะถูกเขียนทับอีกครั้งใน Step 6 หลังผ่าน Step 2–5 ครบ ไฟล์ที่เหลือจบที่นี่ (ไม่มีขั้นตอนถัดไปแตะต้อง)

## 2. Harmonize category taxonomy (ก่อน dedup)

## 3. Resolve dates — CHECKPOINT

## 4. Deduplicate

## 5. Remaining field-level fixes

## 6. Validate + write staging